<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания: 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.

Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.


----



#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
public interface IRefundable
{
    int RefundDays { get; set; }
    bool Refund(decimal amount);
}
public interface ILoggable
{
    void LogOperation(string message);
    void ShowLog();
}



public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set
        {
            if (value <= 0)
                throw new ArgumentException("ID должен быть больше 0.");
            _paymentMethodId = value;
        }
    }
    public string MethodName
    {
        get { return _methodName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название не может быть пустым.");
            _methodName = value;
        }
    }
    public decimal MinAmount
    {
        get { return _minAmount; }
        set
        {
            if (value < 0)
                throw new ArgumentException("Минимальная сумма не может быть отрицательной.");
            _minAmount = value;
        }
    }

    public PaymentMethod(int paymentMethodId, string methodName, decimal minAmount)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
    }

    public virtual bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;

        Console.WriteLine($"Платёж на сумму {amount} через {MethodName}.");
        return true;
    }
    public virtual bool CheckMinimumAmount(decimal amount)
    {
        bool isValid = amount >= MinAmount;
        if (isValid)
        {
            Console.WriteLine($"Сумма {amount} соответствует минимальной сумме {MinAmount}.");
        }
        else
        {
            Console.WriteLine($"Сумма {amount} меньше минимальной суммы {MinAmount}.");
        }
        return isValid;
    }
    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID: {PaymentMethodId}, Название: {MethodName}, Минимальная сумма: {MinAmount}");
    }
    public virtual decimal CalculateFee(decimal amount) => 0m;
}

public class OnlinePayment : PaymentMethod, IRefundable, ILoggable
{
    private string _paymentUrl;
    private int _refundDays;
    private decimal _totalPaid;
    private List<string> _log = new List<string>();

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set
        {
            if (string.IsNullOrWhiteSpace(value) || !value.StartsWith("http"))
                throw new ArgumentException("URL должен начинаться с http.");
            _paymentUrl = value;
        }
    }

    public int RefundDays
    {
        get { return _refundDays; }
        set
        {
            if (value < 0)
                throw new ArgumentException("Количество дней для возврата не может быть отрицательным.");
            _refundDays = value;
        }
    }

    public decimal TotalPaid
    {
        get { return _totalPaid; }
    }

    public bool Refund(decimal amount)
    {
        if (amount <= 0 || amount > _totalPaid)
        {
            Console.WriteLine($"Возврат {amount} невозможен. Доступно для возврата: {_totalPaid}");
            LogOperation($"Отказ в возврате {amount}");
            return false;
        }

        _totalPaid -= amount;
        Console.WriteLine($"Возврат {amount} через {MethodName}. Срок возврата: {RefundDays} дн.");
        LogOperation($"Возврат {amount}");
        return true;
    }
    public void LogOperation(string message)
    {
        _log.Add($"[{DateTime.Now:HH:mm:ss}] {message}");
    }
    public void ShowLog()
    {
        Console.WriteLine($"Журнал {MethodName}:");
        foreach (string entry in _log)
            Console.WriteLine($"  {entry}");
    }

    public OnlinePayment(int paymentMethodId, string methodName, decimal minAmount, string paymentUrl, int refundDays)
        : base(paymentMethodId, methodName, minAmount)
    {
        PaymentUrl = paymentUrl;
        RefundDays = refundDays;
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;
        _totalPaid += amount;
        Console.WriteLine($"Онлайн оплата на сумму {amount} по ссылке {PaymentUrl}.");
        LogOperation($"Оплата {amount}");   
        return true;
    }
}

public class BankTransfer : PaymentMethod
{
    private string _bankData;
    private const decimal TransferFee = 0.1m;

    public string BankData
    {
        get { return _bankData; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Банковские данные не могут быть пустыми.");
            _bankData = value;
        }
    }

    public BankTransfer(int paymentMethodId, string methodName, decimal minAmount, string bankData)
        : base(paymentMethodId, methodName, minAmount)
    {
        BankData = bankData;
    }
    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal amountAfterFee = amount - amount * TransferFee;
        Console.WriteLine($"Комиссия {TransferFee * 100}%, сумма после комиссии: {amountAfterFee}");
        return base.CheckMinimumAmount(amountAfterFee);
    }

    public override decimal CalculateFee(decimal amount) => amount * TransferFee;

    public override bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;

        decimal fee = CalculateFee(amount);
        Console.WriteLine($"Банковский перевод на сумму {amount}. Комиссия: {fee} ({TransferFee * 100}%), получатель получит {amount - fee}.");
        return true;
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Банковские данные: {BankData}, комиссия: {TransferFee * 100}%");
    }
}

public class InternationalTransfer : BankTransfer
{
    private string _swiftCode;
    private const decimal InternationalFee = 15m;
    private string _destinationCountry;


    public string SwiftCode
    {
        get { return _swiftCode; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("SWIFT код не может быть пустым.");
            _swiftCode = value;
        }
    }

    public string DestinationCountry
    {
        get { return _destinationCountry; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Страна назначения не может быть пустой.");
            _destinationCountry = value;
        }
    }

    public InternationalTransfer(int paymentMethodId, string methodName, decimal minAmount, string bankData, string swiftCode, string destinationCountry)
        : base(paymentMethodId, methodName, minAmount, bankData)
    {
        SwiftCode = swiftCode;
        DestinationCountry = destinationCountry;
    }

    public override decimal CalculateFee(decimal amount)
    {
        return base.CalculateFee(amount) + InternationalFee;
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;
        decimal fee = CalculateFee(amount);
        Console.WriteLine($"Международный перевод на сумму {amount} в страну: {DestinationCountry}.");
        Console.WriteLine($"Общая комиссия: {fee}, получатель получит {amount - fee}.");
        return true;
    }


    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"SWIFT код: {SwiftCode}. Страна назначения: {DestinationCountry}");
    }
}


public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Место выдачи не может быть пустым.");
            _cashPickupPoint = value;
        }
    }
    public CashPayment(int paymentMethodId, string methodName, decimal minAmount, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount)
    {
        CashPickupPoint = cashPickupPoint;
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Место получения наличных: {CashPickupPoint}");
    }
}


public class Shop
{
    private string _name;
    private decimal _revenue;

    public string Name
    {
        get { return _name; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название магазина не может быть пустым.");
            _name = value;
        }
    }

    public decimal Revenue
    {
        get { return _revenue; }
        private set { _revenue = value; }
    }

    public Shop(string name)
    {
        Name = name;
    }

    public void Receive(decimal amount)
    {
        Revenue += amount;
        Console.WriteLine($"Магазин {Name} получил {amount}. Выручка: {Revenue}");
    }
}


public class Customer
{
    private string _name;
    private decimal _balance;
    private List<PaymentMethod> _paymentMethods = new List<PaymentMethod>();

    public string Name
    {
        get { return _name; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя не может быть пустым.");
            _name = value;
        }
    }

    public decimal Balance
    {
        get { return _balance; }
        private set
        {
            if (value < 0)
                throw new ArgumentException("Баланс не может быть отрицательным.");
            _balance = value;
        }
    }

    public Customer(string name, decimal balance)
    {
        Name = name;
        Balance = balance;
    }

    public void AddPaymentMethod(PaymentMethod method)
    {
        if (method == null)
            throw new ArgumentNullException(nameof(method));
        _paymentMethods.Add(method);
        Console.WriteLine($"{Name} добавил способ оплаты: {method.MethodName}");
    }

    public bool Pay(Shop shop, PaymentMethod method, decimal amount)
    {
        Console.WriteLine($"\n--- {Name} платит {amount} в магазине {shop.Name} ({method.MethodName}) ---");

        if (!_paymentMethods.Contains(method))
        {
            Console.WriteLine("Этот способ оплаты не подключён у клиента.");
            return false;
        }

        if (amount > Balance)
        {
            Console.WriteLine($"Недостаточно средств. Баланс: {Balance}");
            return false;
        }

        if (!method.ProcessPayment(amount))
        {
            Console.WriteLine("Платёж отклонён.");
            return false;
        }

        decimal fee = method.CalculateFee(amount);

        Balance -= amount;
        shop.Receive(amount - fee);
        Console.WriteLine($"Платёж прошёл. Баланс {Name}: {Balance}");
        return true;
    }
}


In [17]:
Shop shop = new Shop("EuroShop");
Customer customer = new Customer("Иван", 5000m);

OnlinePayment online = new OnlinePayment(1, "Онлайн оплата", 50m, "https://pay.example.com", 14);
CashPayment cash = new CashPayment(3, "Наличные", 10m, "Касса №1");

customer.AddPaymentMethod(online);
customer.AddPaymentMethod(cash);

customer.Pay(shop, online, 500m);
customer.Pay(shop, cash, 100m);

Console.WriteLine("\n--- Возвраты ---");
online.Refund(200m);     // пройдёт
online.Refund(1000m);    // отказ: больше, чем оплачено

Console.WriteLine();
online.ShowLog();

Console.WriteLine("\n--- Проверка типа ---");
PaymentMethod[] methods = { online, cash };
foreach (PaymentMethod m in methods)
{
    if (m is IRefundable)
        Console.WriteLine($"{m.MethodName}: возврат поддерживается");
    else
        Console.WriteLine($"{m.MethodName}: возврат не поддерживается");
}

Иван добавил способ оплаты: Онлайн оплата
Иван добавил способ оплаты: Наличные

--- Иван платит 500 в магазине EuroShop (Онлайн оплата) ---
Сумма 500 соответствует минимальной сумме 50.
Онлайн оплата на сумму 500 по ссылке https://pay.example.com.
Магазин EuroShop получил 500. Выручка: 500
Платёж прошёл. Баланс Иван: 4500

--- Иван платит 100 в магазине EuroShop (Наличные) ---
Сумма 100 соответствует минимальной сумме 10.
Платёж на сумму 100 через Наличные.
Магазин EuroShop получил 100. Выручка: 600
Платёж прошёл. Баланс Иван: 4400

--- Возвраты ---
Возврат 200 невозможен. Доступно для возврата: 0
Возврат 1000 невозможен. Доступно для возврата: 0

Журнал Онлайн оплата:
  [12:02:15] Отказ в возврате 200
  [12:02:15] Отказ в возврате 1000

--- Проверка типа ---
Онлайн оплата: возврат поддерживается
Наличные: возврат не поддерживается
